# <font color="#003660">Applied Machine Learning for Text Analysis (M.184.5331)</font>


# <font color="#003660">Session 2: Transformer Architecture - Encoder-only Models</font>

# <font color="#003660">Notebook 1: A Tour of Hugging Face Transformers</font>

<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/dag.png"/><br></center>

<p>
<center>
<div>
    <font color="#085986"><b>By the end of this lesson, you ...</b><br><br>
        ... will have a high-level understanding of the Transformer architecture, <br>
        ... will know the basic types of Transformers (i.e., encoder, decoder, encoder-decoder), and <br>
        ... will know how to use pre-trained NLP pipeline (e.g., sentiment analysis, NER, translation) from the Hugging Face Transformers library.
    </font>
</div>
</center>
</p>

The following content is heavily inspired by the following excellent sources:


*   Tunstall et al. (2021): Natural Language Processing with Transformers. O'Reilly. https://www.oreilly.com/library/view/natural-language-processing/9781098103231/
*   Hugging Face (2021): Transformer Models - Hugging Face Course. https://huggingface.co/course/



# What are Transformers?

## Overview

The general **Transformer** architecture consists of two components:

*   **Encoder**: The encoder processes an input sequence and builds a numerical representation (feature vector) of it. This means that this component is optimized for understanding the input.
*   **Decoder**: The decoder processes the feature vector produced by the encoder, plus other inputs, to generate an output sequence. This means that this component is optimized for generating outputs.







<center><br><img width=256 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/architecture_high-level.png"/><br></center>

Depending on the task at hand, the components of the Transformer architecture can be used independently or in combination:

*   **Encoders**: Good for tasks that require understanding of the input, such as *text classification* and *named entity recognition*. Watch [this video](https://youtu.be/MUqNwgPjJvQ) to learn more.
*   **Decoders**: Good for generative tasks such as *text generation*. Watch [this video](https://youtu.be/d_ixlCubqQw) to learn more.
*   **Encoder-decoder Models** (aka sequence-to-sequence models): Good for generative tasks that require an input, such as *translation* or *summarization*. Watch [this video](https://youtu.be/0_4KEb08xrE) to learn more.







<center><br><img width=500 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/transformer_models.png"/><br></center>

## Attention is all you need!

In general, the encoder and decoder components can be any kind of neural network architecture that is suited for modeling sequences. For example, in the diagram below, simple recurrent neural networks (RNN) are used to implement the encoder and decoder components.

In this example, the English sentence “Transformers are great!” is encoded into a hidden state vector that is then decoded to produce the German translation “Transformer sind grossartig!” (Note: besides the hidden state, the decoder normally also uses the already generated output tokens as an additional input).

Note how this network processes inputs and generates outputs in a sequential way, indicated the vertical lines between the RNN cells. This sequential processing is slow, as it cannot take full advantage of the parallel processing capabilities of a GPU.

<center><br><img width=600 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/transformer_rnn.png"/><br></center>

One weakness with the above architecture is that only the final hidden state of the encoder is passed to the decoder, which creates an information bottleneck. The meaning of the whole input sequence has to be captured in in just one vector, which is especially challenging for long sequences where information at the start of the sequence might be lost in the process of creating a single vector representation.

Of course, a straight forward way to avoid this bottleneck is to pass all of the encoder’s hidden states to the decorder (not shown in the diagram). This way, almost no information would be lost. Yet, at the same time we might pass a lot of irrelevant information to the decoder. 


Ideally, we would select which information in the input sequence is relevant for producing a specific part of the output sequence. **Attention** is a technique that exactly allows this.

The figure below visualizes the attention weights for an English-to-French translation model. Note how the decoder is able to correctly align the words “zone” and “Area”, which are ordered differently in the two languages.

<center><br><img width=500 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/att_translation.png"/><br></center>

Attention can also be used to encode input sequences. The figure belows how the embedding of the token "it" takes the embedding vectors of previous words in the sequence (esp. "robot") into account.

<center><img width=600 src="https://raw.githubusercontent.com/olivermueller/amlta-2026/main/Session_02/self_attention.png"/></center>

In the famous "[Attention is all you need paper](http://papers.nips.cc/paper/7181-attention-is-all-you-%0Aneed.pdf)", Vaswani et al. showed that the RNNs inside the encoder and decoder components can be replaced entirely with Attention and feed-forward layers.

This architecture has three main advantages:

* First, without RNN cells, all tokens can be fed in parallel through the model, which makes the model faster and allows to train it on larger corpora.

* Second, the Attention mechanism makes the network more effective on tasks that require memorizing information over long time sequences.

* Third, the Attention mechanisms creates a representation for each token that is dependent on its surrounding tokens. This makes the representation of each token context aware, such that the representation of the word “apple” (fruit) is different from “apple” (computer manufacturer).

<center><br><img width=600 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/transformer_att2.png"/><br></center>

In modern Transformer architectures, several blocks of Attention and feed-forward layers are stacked in the encoder to produce rich hidden states which are then passed to the decoder.

This information should be enough to build an intuitive understanding of Attention. If you want to learn more about its technical details, read Chapter 3 of Tunstall et al. (2021).

## Transfer Learning

Another feature of Transformers is this use of transfer learning. Transfer learning is a machine learning approach that involves applying knowledge gained from solving one problem to solve a different but related problem.

This usually works by splitting a model in terms of a **body** and **head**. During pretraining, the weights of the body are optimized to represent broad features of the source domain (e.g., vocabulary, grammar). These weights are then used to initialize the new model for the new task. The head is a task-specific network that is only trained during fine-tuning.

The figure below illustrates this idea and contrasts it with traditional machine learning. For example, in the models on the right Body A could be pretrained with a language modeling task in Domain A and the used to initialize Body B of a sentiment analysis model in Domain B. The weights of Head B, the sentiment classifier, are learned from scratch.

<center><br><img width=700 src="https://raw.githubusercontent.com/olivermueller/aml4ta-2021/main/resources/transfer_learning.png"/><br></center>

Transfer learning typically produces models that can be fine-tuned efficiently on a variety of downstream tasks (i.e., with less time and data).

# Import Packages

As always, we first need to load a number of required Python packages:
- `pandas` provides high-performance, easy-to-use data structures and data analysis tools.
- `numpy` is a library adding support for large, multi-dimensional arrays and matrices, along with a large collection of high-level mathematical functions to operate on these arrays.
- `sklearn` is a free software machine learning library for the Python programming language.
- `transformers` provides general-purpose architectures (BERT, GPT-2, RoBERTa, XLM, DistilBert, XLNet…) for Natural Language Understanding (NLU) and Natural Language Generation (NLG) with over 32+ pretrained models in 100+ languages.

In [ ]:
import pandas as pd
import numpy as np
import torch
from sklearn import metrics
from transformers import pipeline, set_seed
from transformers import AutoTokenizer, AutoModelForQuestionAnswering, AutoModelForSeq2SeqLM

In [ ]:
import warnings
warnings.filterwarnings("ignore")

set_seed(42)

# Pre-trained Pipelines

In the following, we will use some pre-trained NLP pipelines from the Hugging Face Transformers library.

These pipelines are an easy way to use models for inference. They abstract most of the complex code from the library (e.g., code for tokenizing input sequences or for training loops), offering a simple API dedicated to specific tasks. For more, see: https://huggingface.co/docs/transformers/main_classes/pipelines

We deliberately use *small* models (most of them about 250-300 MB), so that they download quickly and run on any laptop. Larger models are usually more accurate.

## Sentiment Analysis

The sentiment analysis pipeline uses a model that was fine-tuned on the Stanford Sentiment Treebank, which is an English corpus of annotated movie reviews. This is an example of an encoder model.

In [ ]:
sa_classifier = pipeline("sentiment-analysis", model="distilbert/distilbert-base-uncased-finetuned-sst-2-english")

In [ ]:
sa_classifier([
            "This is the best movie I have ever seen.",
            "I hate this movie so much!",
            "I don't like the new iPhone.",
            "Becks is not bad.",
     ])

## Named Entity Recognition

Named entities are names of products, places or people and detecting and extracting them from text is called named entity recognition (NER). This is an example of an encoder model.

In [ ]:
ner = pipeline("ner", model="dslim/distilbert-NER", aggregation_strategy="first")  # merge sub-word tokens into whole words

In [ ]:
ner("My name is Oliver and I work at Paderborn University.")

## Text Generation

Text generation is the task of generating text with the goal of appearing indistinguishable to human-written text. This is an example of a decoder model.

In [ ]:
txt_generator = pipeline("text-generation", model="HuggingFaceTB/SmolLM2-135M-Instruct")

In [ ]:
set_seed(42)  # text generation samples randomly; fix the seed to get reproducible texts
txt_generator("In this course, you will learn how to",
              num_return_sequences = 2,
              max_new_tokens = 40,
              do_sample = True)

# One Chat Model for Many Tasks

Until recently, every NLP task had its own pipeline with a specialized model, e.g., for summarization, translation, or question answering. Since version 5 of the Transformers library, these pipelines have been removed. Instead, Hugging Face recommends to solve such tasks with a single **chat model**: a decoder model (like the one we used for text generation above) that was additionally trained to follow instructions. We simply describe the task in natural language (the *prompt*), and the model generates the answer. You will learn much more about chat models and prompting in Session 4.

Chat models expect a conversation, i.e., a list of messages, each with a `role` (`"user"` for our input, `"assistant"` for the model's replies) and a `content`. We use the `text-generation` pipeline with a small chat model, Qwen2.5-0.5B-Instruct (about 1 GB). If you run this notebook on a GPU (e.g., on Colab), try a larger model like `Qwen/Qwen2.5-1.5B-Instruct` for better answers.

In [ ]:
chat = pipeline("text-generation", model="Qwen/Qwen2.5-0.5B-Instruct")

## Question Answering

There are two fundamentally different ways to answer a question with a language model. We look at both of them.

### Extractive Question Answering: The Answer Is in the Context

In *extractive* question answering, we provide the model with a passage of text, called the context, along with a question about it. The answer can be found in the context, and the model's job is essentially reading comprehension.

Let's send our first message to the chat model and look at the raw output.

In [ ]:
context = "My name is Oliver and I work at Paderborn University."
messages = [{"role": "user", "content": f"Answer the question based on the context.\n\nContext: {context}\n\nQuestion: Where do I work?"}]

chat(messages, max_new_tokens=60)

The pipeline returns the whole conversation, with the model's reply as the last message. For convenience, we write a small function that sends a prompt to the model and returns only the content of its reply.

In [ ]:
def ask(prompt, max_new_tokens=100):
    set_seed(42)  # chat models sample their answers randomly; fix the seed to get reproducible answers
    messages = [{"role": "user", "content": prompt}]
    return chat(messages, max_new_tokens=max_new_tokens)[0]["generated_text"][-1]["content"]

In [ ]:
print(ask(f"Answer the question based on the context.\n\nContext: {context}\n\nQuestion: Where do I work?"))

### Open Question Answering: The Answer Is in the Model's Weights

In *open* (or closed-book) question answering, we ask a question without providing any context. The model has to answer from the knowledge it picked up from its training data during pre-training, which is stored in its weights.

In [ ]:
print(ask("What is a transformer model? Answer in one sentence."))

Transformers are a popular topic, so the model has seen many texts about them during pre-training. But what happens if we ask it something it cannot know? Let's ask our extractive question again – this time without the context.

In [ ]:
print(ask("Where does Oliver work?"))

What's the difference?

| | Extractive QA | Open QA |
|---|---|---|
| Source of the answer | the context we provide | knowledge stored in the model's weights |
| Works for private or new information? | yes – the model has never heard of Oliver, but finds the answer in the context | no – the model only knows what was in its training data (and only up to its training cutoff) |
| Can we check the answer? | yes, against the context | hardly – the model doesn't tell us where its knowledge comes from |
| Typical failure | misreading the context | *hallucination*: a fluent, confident, but wrong answer (more likely for small models and rare facts) |

The model has never heard of our Oliver, but instead of admitting that, it invents a plausible-sounding answer. Combining both approaches – first search for relevant documents, then answer the question based on them – is the idea behind *retrieval-augmented generation* (RAG), which you will meet in Session 5.

## Summarization

The goal of text summarization is to take a long text as input and generate a short version with all relevant facts.

In [ ]:
text = """Darth Vader is a fictional character in the Star Wars franchise. The character is the primary antagonist
in the original trilogy and, as Anakin Skywalker, is a primary protagonist in the prequel trilogy. Star Wars creator
George Lucas has collectively referred to the first six episodic films of the franchise as "the tragedy of Darth Vader".
He has become one of the most iconic villains in popular culture, and has been listed among the greatest villains and
fictional characters ever."""

print(ask(f"Summarize the following text in one sentence:\n\n{text}"))

## Translation

Translation is the task of translating text or speech from one language to another.

In [ ]:
print(ask("Translate the following French text into English:\n\nLa bibliothèque Hugging Face est tout simplement géniale."))

Now it's your turn: Try other tasks (e.g., sentiment analysis or named entity recognition) by changing the prompt. Where does the small chat model fail?

In [ ]:
# YOUR CODE GOES HERE

# Task-specific Models

Chat models are flexible, but large. For many tasks, there are also small, specialized models that were fine-tuned for exactly one task. Even without a pipeline, we can still use them: tokenize the input, feed it through the model, and decode the output. This is also what a pipeline does behind the scenes.

## Translation with an Encoder-Decoder Model

This model was trained only to translate from French to English (about 300 MB). It is an example of an encoder-decoder model: the encoder reads the French text, the decoder generates the English translation.

In [ ]:
tr_model_name = "Helsinki-NLP/opus-mt-fr-en"
tr_tokenizer = AutoTokenizer.from_pretrained(tr_model_name)
tr_model = AutoModelForSeq2SeqLM.from_pretrained(tr_model_name)

In [ ]:
inputs = tr_tokenizer("La bibliothèque Hugging Face est tout simplement géniale.", return_tensors="pt")
output_ids = tr_model.generate(**inputs)
tr_tokenizer.decode(output_ids[0], skip_special_tokens=True)

## Extractive Question Answering with an Encoder Model

This model (about 260 MB) does not generate an answer. Instead, it *extracts* the answer from the context: it is an encoder model that predicts, for every token of the context, how likely the answer starts and ends there.

In [ ]:
qa_model_name = "distilbert/distilbert-base-cased-distilled-squad"
qa_tokenizer = AutoTokenizer.from_pretrained(qa_model_name)
qa_model = AutoModelForQuestionAnswering.from_pretrained(qa_model_name)

In [ ]:
question = "Where do I work?"
context = "My name is Oliver and I work at Paderborn University."

inputs = qa_tokenizer(question, context, return_tensors="pt")
with torch.no_grad():
    outputs = qa_model(**inputs)

# the answer is the span between the most likely start token and the most likely end token
start = outputs.start_logits.argmax()
end = outputs.end_logits.argmax()
qa_tokenizer.decode(inputs["input_ids"][0][start:end + 1])